In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
from pymatgen.io.lobster.outputs import Icohplist
from lobsterpy.cohp.analyze import Analysis
from lobsterpy.featurize.utils import get_file_paths
from lobsterpy.plotting import (
    PlainCohpPlotter,
    IcohpDistancePlotter,
    InteractiveCohpPlotter,
)

warnings.filterwarnings("ignore")

In [ ]:
calc_dir = "/home/anaik/RZ-Dienste/AG-Jgeorge/anaik/For_Joana/RT_Ag8XXS6/Ag8SnS6"

In [ ]:
# get file paths
req_files = get_file_paths(
    path_to_lobster_calc=calc_dir,
    requested_files=[
        "structure",
        "charge",
        "icohplist",
        "cohpcar",
        "icobilist",
        "cobicar",
        "madelung",
    ],
)

# Plot ICOHPs and ICOBIs (Fig 2 (a) and (c))

In [ ]:
icobis = Icohplist(filename=req_files.get("icobilist"), are_cobis=True, are_coops=False)
icohps = Icohplist(
    filename=req_files.get("icohplist"), are_cobis=False, are_coops=False
)

In [ ]:
for icoxx in [icobis, icohps]:
    plotter = IcohpDistancePlotter(are_cobis=icoxx.are_cobis, are_coops=icoxx.are_coops)
    plotter.add_icohps(label="Ag8SnS6", icohpcollection=icoxx.icohpcollection)
    plotter.get_plot(color_interactions=True, marker_size=30, colors=None, alpha=1)

# Analyze COHPs (Fig 2(b))

In [ ]:
analyze_cohps = Analysis(
    path_to_charge=req_files.get("charge"),
    path_to_cohpcar=req_files.get("cohpcar"),
    path_to_poscar=req_files.get("structure"),
    path_to_icohplist=req_files.get("icohplist"),
    path_to_madelung=req_files.get("madelung"),
    orbital_resolved=True,
    which_bonds="cation-anion",
)

In [ ]:
analyze_cohps.condensed_bonding_analysis  # get overview of most important bonds and orbitals

In [ ]:
# dict with most relevant orbitals for Ag-S bonds, key is orbital pair, value is plot legend label
max_contributing_orb = {"3p-5s": "Ag(5s)-S(3p)", "3p-4d": "Ag(4d)-S(3p)"}

## Plot orbital resolved COHPs of Ag-S bonds

In [ ]:
cohp_plotter = PlainCohpPlotter()
for rel_orb in max_contributing_orb:
    orb_list = []  # list to store orbitals needed for plotting
    label_list = []  # list to store labels of Ag-S bonds
    for bond, rel_data in analyze_cohps.get_site_orbital_resolved_labels().items():
        if "Ag-S" in bond:
            for orb, value in rel_data.items():
                if rel_orb == orb:
                    for label in value["bond_labels"]:
                        mapped_bond_labels = [
                            item
                            for item in [label]
                            for _ in range(len(value["relevant_sub_orbitals"]))
                        ]
                        orb_list.extend(value["relevant_sub_orbitals"])
                        label_list.extend(mapped_bond_labels)

    cohp_orb = (
        analyze_cohps.chemenv.completecohp.get_summed_cohp_by_label_and_orbital_list(
            label_list=label_list, orbital_list=orb_list, divisor=len(set(label_list))
        )
    )

    summed_cohp = analyze_cohps.chemenv.completecohp.get_summed_cohp_by_label_list(
        label_list=list(set(label_list)), divisor=len(set(label_list))
    )

    cohp_plotter.add_cohp(label="Ag-S", cohp=summed_cohp)
    cohp_plotter.add_cohp(label=max_contributing_orb.get(rel_orb), cohp=cohp_orb)


cohp_plotter.get_plot(sigma=0.05, ylim=(-10, 5));

# Analyze COBIs (Fig 2(d))

In [ ]:
analyze_cobis = Analysis(
    path_to_charge=req_files.get("charge"),
    path_to_cohpcar=req_files.get("cobicar"),
    path_to_poscar=req_files.get("structure"),
    path_to_icohplist=req_files.get("icobilist"),
    path_to_madelung=req_files.get("madelung"),
    orbital_resolved=True,
    are_cobis=True,
    noise_cutoff=0.001,
    which_bonds="cation-anion",
)

In [ ]:
analyze_cobis.condensed_bonding_analysis  # get overview of most important bonds and orbitals

In [ ]:
# dict with most relevant orbitals for Ag-S bonds, key is orbital pair, value is plot legend label
max_contributing_orb = {"3p-5s": "Ag(5s)-S(3p)", "3p-4d": "Ag(4d)-S(3p)"}

## Plot orbital resolved COBIs of Ag-S bonds

In [ ]:
cobi_plotter = PlainCohpPlotter(are_cobis=True)
for rel_orb in max_contributing_orb:
    orb_list = []  # list to store orbitals needed for plotting
    label_list = []  # list to store labels of Ag-S bonds
    for bond, rel_data in analyze_cobis.get_site_orbital_resolved_labels().items():
        if "Ag-S" in bond:
            for orb, value in rel_data.items():
                if rel_orb == orb:
                    for label in value["bond_labels"]:
                        mapped_bond_labels = [
                            item
                            for item in [label]
                            for _ in range(len(value["relevant_sub_orbitals"]))
                        ]
                        orb_list.extend(value["relevant_sub_orbitals"])
                        label_list.extend(mapped_bond_labels)

    cobi_orb = (
        analyze_cobis.chemenv.completecohp.get_summed_cohp_by_label_and_orbital_list(
            label_list=label_list, orbital_list=orb_list, divisor=len(set(label_list))
        )
    )

    summed_cobi = analyze_cobis.chemenv.completecohp.get_summed_cohp_by_label_list(
        label_list=list(set(label_list)), divisor=len(set(label_list))
    )

    cobi_plotter.add_cohp(label="Ag-S", cohp=summed_cobi)
    cobi_plotter.add_cohp(label=max_contributing_orb.get(rel_orb), cohp=cobi_orb)


cobi_plotter.get_plot(sigma=0.05, ylim=(-10, 5));